# Delta Lake Tables
Use this notebook to explore Delta Lake functionality 


In [1]:
from pyspark.sql.types import StructType, IntegerType, StringType, DoubleType

# define the schema

schema = StructType()\
.add("ProductID", IntegerType(), True) \
.add("ProductName", StringType(), True) \
.add("Category", StringType(), True) \
.add("ListPrice", DoubleType(), True)

df = spark.read.format("csv").option("header", True).schema(schema).load("Files/Products/products.csv")
# df now is a Spark DataFrame containing CSV data from "Files/products/products.csv".
display(df)

StatementMeta(, 525255f7-2ff4-433b-9894-2873d491b43c, 3, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f12a765b-ee2a-4d2b-a6e2-8f72e2cf2e6a)

In [6]:
# To create a managed Delta table
df.write.format("delta").saveAsTable("managed_products")

StatementMeta(, 928e45ac-d87d-4ff4-bc98-127d49d6ca54, 8, Finished, Available, Finished, False)

In [7]:
# Create an external table

df.write.format("delta").saveAsTable("external_products", path="abfss://2fef5321-df19-4be3-bd55-686e54fc465e@onelake.dfs.fabric.microsoft.com/54e07f84-a6f4-4f18-9627-8334674c4f86/Files/external_products")


StatementMeta(, 928e45ac-d87d-4ff4-bc98-127d49d6ca54, 9, Finished, Available, Finished, False)

In [10]:
%%sql
-- Compare managed and external tables
-- managed tables

DESCRIBE FORMATTED managed_products

StatementMeta(, 928e45ac-d87d-4ff4-bc98-127d49d6ca54, 12, Finished, Available, Finished, False)

<Spark SQL result set with 12 rows and 3 fields>

In [11]:
%%sql
-- external tables

DESCRIBE FORMATTED external_products


StatementMeta(, 928e45ac-d87d-4ff4-bc98-127d49d6ca54, 13, Finished, Available, Finished, False)

<Spark SQL result set with 12 rows and 3 fields>

In [2]:
%%sql

-- Drop tables

DROP TABLE managed_products;
DROP TABLE external_products;

StatementMeta(, 525255f7-2ff4-433b-9894-2873d491b43c, 5, Finished, Available, Finished, True)

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

In [3]:
%%sql

-- create a Delta table, using the %%sql magic command.
CREATE TABLE products
USING DELTA
LOCATION "Files/external_products";

StatementMeta(, 525255f7-2ff4-433b-9894-2873d491b43c, 6, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [5]:
%%sql

-- explore products table

SELECT *
FROM products
LIMIT 5;

StatementMeta(, 525255f7-2ff4-433b-9894-2873d491b43c, 8, Finished, Available, Finished, False)

<Spark SQL result set with 5 rows and 4 fields>

In [9]:
%%sql

-- table versioning
-- implement a 10% reduction in the price for mountain bikes

UPDATE products
SET ListPrice = ListPrice * 0.9
WHERE Category = "Mountain Bikes";

StatementMeta(, 525255f7-2ff4-433b-9894-2873d491b43c, 12, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [10]:
%%sql
-- transaction log on production table

DESCRIBE HISTORY products;

StatementMeta(, 525255f7-2ff4-433b-9894-2873d491b43c, 13, Finished, Available, Finished, False)

<Spark SQL result set with 2 rows and 15 fields>

In [11]:
%%sql

SELECT *
FROM Products 
LIMIT 5;

StatementMeta(, 525255f7-2ff4-433b-9894-2873d491b43c, 14, Finished, Available, Finished, False)

<Spark SQL result set with 5 rows and 4 fields>

In [16]:
delta_table_path = "Files/external_products"
# Get the current data

current_data = spark.read.format("delta").load(delta_table_path)
display(current_data.limit(1))

# Get the version 0 data

original_data = spark.read.format("delta").option("versionAsOf", 0).load(delta_table_path)
display(original_data.limit(1))

StatementMeta(, 525255f7-2ff4-433b-9894-2873d491b43c, 19, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 283a33f4-aeec-4689-aa9d-6766c380788d)

SynapseWidget(Synapse.DataFrame, 3e098292-4e73-40dd-b108-b7ad5879fd87)

In [17]:
%%sql
-- Analyze Delta table data with SQL queries

-- create a temporary view
CREATE OR REPLACE TEMPORARY VIEW products_view
AS
SELECT Category, Count(*) AS NumProducts, MIN(ListPrice) AS MinPrice, MAX(ListPrice) AS MAxPrice, AVG(ListPrice) AS AvgPrice
FROM products 
GROUP BY Category;

SELECT *
FROM products_view
ORDER BY Category;

StatementMeta(, 525255f7-2ff4-433b-9894-2873d491b43c, 21, Finished, Available, Finished, True)

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 37 rows and 5 fields>

In [19]:
%%sql

-- top 10 categories by number of products

SELECT Category, NumProducts
FROM products_view
ORDER BY NumProducts DESC
LIMIT 10;

StatementMeta(, 525255f7-2ff4-433b-9894-2873d491b43c, 23, Finished, Available, Finished, False)

<Spark SQL result set with 10 rows and 2 fields>

In [23]:
# SQL query using PySpark.

from pyspark.sql.functions import col, desc

df_products = spark.sql(
"SELECT Category, MinPrice, MaxPrice, AvgPrice \
FROM products_view") \
.orderBy(col("AvgPrice") \
.desc()
)

display(df_products.limit(5))


StatementMeta(, 525255f7-2ff4-433b-9894-2873d491b43c, 27, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b5589b24-cace-4488-aecc-b403bbaf8062)

In [30]:
# Delta tables for streaming data

## Delta table as a sink for some streaming data in a simulated internet of things (IoT) scenario.

from notebookutils import mssparkutils # Microsoft Spark Utilities
from pyspark.sql.types import *
from pyspark.sql.functions import *

# create a folder
input_path = "Files/data/"
mssparkutils.fs.mkdirs(input_path)

# Create a stream that reads data from the folder, using a JSON schema
jsonSchema = StructType([
    StructField("device", StringType(), False), # "False" - field cannot contain NULL values
    StructField("status", StringType(), False)
])

iotstream = spark.readStream.schema(jsonSchema).option("maxFilesPerTrigger", 1).json(input_path)

# Write some event data to the folder
device_data = '''{"device": "Dev1", "status":"ok"}
{"device":"Dev1", "status":"ok"}
{"device":"Dev1", "status":"ok"}
{"device":"Dev2", "status":"error"}
{"device":"Dev1", "status":"ok"}
{"device":"Dev1", "status":"error"}
{"device":"Dev2", "status":"ok"}
{"device":"Dev2", "status":"error"}
{"device":"Dev1", "status":"ok"}'''

mssparkutils.fs.put(input_path + "data.txt", device_data, True)

print("Source stream created...")  

''' created a streaming data source based on a folder 
    to which some data has been saved, representing 
    readings from hypothetical IoT devices.'''

StatementMeta(, 525255f7-2ff4-433b-9894-2873d491b43c, 34, Finished, Available, Finished, False)

Source stream created...


' created a streaming data source based on a folder \n    to which some data has been saved, representing \n    readings from hypothetical IoT devices.'

In [37]:
# Write the stream to a delta table

data_steam_table_path = "Tables/iotdevicedata"
checkpointpath = "Files/delta/checkpoint"

deltastream = iotstream.writeStream.format("delta").option("checkpointLocation", checkpointpath).start(data_steam_table_path)

print("Streaming to delta sink...")

'''writes the streaming device data in Delta format to a folder named iotdevicedata. 
Because the path for the folder location in the Tables folder, a table will automatically be created for it.'''

StatementMeta(, 525255f7-2ff4-433b-9894-2873d491b43c, 41, Finished, Available, Finished, False)

Streaming to delta sink...


'writes the streaming device data in Delta format to a folder named iotdevicedata. \nBecause the path for the folder location in the Tables folder, a table will automatically be created for it.'

In [32]:
%%sql
-- explore iotdevicedata table

SELECT *
FROM iotdevicedata

StatementMeta(, 525255f7-2ff4-433b-9894-2873d491b43c, 36, Finished, Available, Finished, False)

<Spark SQL result set with 9 rows and 2 fields>

In [33]:
# Add more data to the source stream
more_data = '''{"device":"Dev1","status":"ok"}
{"device":"Dev1","status":"ok"}
{"device":"Dev1","status":"ok"}
{"device":"Dev1","status":"ok"}
{"device":"Dev1","status":"error"}
{"device":"Dev2","status":"error"}
{"device":"Dev1","status":"ok"}'''

mssparkutils.fs.put(input_path + "more-data.txt", more_data, True)

# writes more hypothetical device data to the streaming source.

StatementMeta(, 525255f7-2ff4-433b-9894-2873d491b43c, 37, Finished, Available, Finished, False)

True

In [38]:
%%sql
SELECT *
FROM iotdevicedata

StatementMeta(, 525255f7-2ff4-433b-9894-2873d491b43c, 42, Finished, Available, Finished, False)

<Spark SQL result set with 16 rows and 2 fields>

In [40]:
deltastream.isActive

StatementMeta(, 525255f7-2ff4-433b-9894-2873d491b43c, 44, Finished, Available, Finished, False)

True

In [42]:
# stop the stream
deltastream.stop()


StatementMeta(, 525255f7-2ff4-433b-9894-2873d491b43c, 46, Finished, Available, Finished, False)